# Stage 5 — Calibration Evaluation

Evaluate cone detection and homography / single-axis calibration on the first frame.

**What this notebook covers:**
- HSV cone detection with current defaults (yellow, orange, blue, red)
- Visualising detected cone centroids overlaid on the first frame
- Running `Calibrator.calibrate_homography` or `calibrate_single_axis`
- Reprojection error check
- World-coordinate mapping: pixel → real-world cm

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH  = Path('../data/raw_footage/sample_clip.mp4')

# Cone world coordinates in cm — adapt to your test setup.
# For explosiveness: no cones needed (single-axis).
# For sprint:        [(0, 0), (500, 0)]   (5 m start/finish)
CONE_WORLD_COORDS_CM = []

from pipeline.ingest import extract_frames
frames_iter = extract_frames(str(VIDEO_PATH), target_fps=15)
_, first_frame, _ = next(frames_iter)
print(f"First frame shape: {first_frame.shape}")

In [ ]:
# ── Inspect HSV range of the first frame ─────────────────────────────────────
hsv = cv2.cvtColor(first_frame, cv2.COLOR_BGR2HSV)
print(f"HSV value ranges in first frame:")
print(f"  H: {hsv[:,:,0].min()}–{hsv[:,:,0].max()}")
print(f"  S: {hsv[:,:,1].min()}–{hsv[:,:,1].max()}")
print(f"  V: {hsv[:,:,2].min()}–{hsv[:,:,2].max()}")

plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1); plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)); plt.title('Original'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(hsv[:,:,0], cmap='hsv'); plt.title('H channel'); plt.colorbar(); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(hsv[:,:,1], cmap='gray'); plt.title('S channel'); plt.colorbar(); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
from pipeline.calibrate import Calibrator

calibrator = Calibrator()

# Detect cones for each default colour
canvas = first_frame.copy()
all_cones = {}
for colour in ('yellow', 'orange', 'blue', 'red'):
    cones = calibrator.detect_cones(first_frame, colour=colour)
    all_cones[colour] = cones
    print(f"  {colour:8s}: {len(cones)} cone(s) detected  {cones}")
    for (cx, cy) in cones:
        cv2.circle(canvas, (int(cx), int(cy)), 12, (0, 255, 255), 2)
        cv2.putText(canvas, colour[0].upper(), (int(cx)+8, int(cy)-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title('Detected Cone Centroids'); plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── Run calibration ───────────────────────────────────────────────────────────
from pipeline.cache import PipelineCache

JOB_ID = 'notebook-eval-detection'
cache  = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))

if CONE_WORLD_COORDS_CM:
    calibration = calibrator.calibrate_homography(first_frame, CONE_WORLD_COORDS_CM)
else:
    calibration = calibrator.calibrate_single_axis(first_frame)

cache.save_calibration(calibration)

print(f"Calibration method : {calibration.method}")
print(f"Valid              : {calibration.is_valid}")
print(f"Reprojection error : {calibration.reprojection_error_px:.3f} px")
print(f"Pixels per cm      : {calibration.pixels_per_cm}")
if calibration.homography_matrix is not None:
    print(f"Homography H:\n{calibration.homography_matrix.round(4)}")

In [ ]:
# ── Test world coordinate mapping ─────────────────────────────────────────────
if calibration.is_valid:
    from pipeline.calibrate import pixel_to_world

    # Test a few pixel positions
    test_pixels = [
        (first_frame.shape[1]//2, first_frame.shape[0]//2),  # centre
        (100, 100),
        (500, 400),
    ]
    print("Pixel → World (cm) mapping:")
    for px, py in test_pixels:
        world = pixel_to_world((float(px), float(py)), calibration)
        if world:
            print(f"  pixel ({px:4d}, {py:4d})  →  world ({world[0]:.1f} cm, {world[1]:.1f} cm)")
        else:
            print(f"  pixel ({px:4d}, {py:4d})  →  mapping unavailable")
else:
    print("Calibration not valid — cannot map pixels to world coordinates.")